Uses Tom's 'stats305c' environment


In [1]:
import dynamax
import numpy as np

I had googled whether dynamax was capable of performing the analysis I wanted, and its AI answered affirmatively so I followed that to see where it went

prompts given:
- can you use python dynamax to learn a low rank observation matrix in a linear dynamical system
- can you show this in code?
- what about where the rank evolves according to a hidden markov model
- can you show the code for this?

Google AI's code:

```python
import jax.numpy as jnp
import jax.random as jr
from jax import jit, value_and_grad
import optax
from dynamax.linear_gaussian_ssm import LinearGaussianSSM

def train_switching_low_rank_lds(emissions, state_dim, ranks, num_iters=300):
    """
    emissions: (T, obs_dim)
    ranks: List of ranks for each discrete state (e.g., [2, 5, 8])
    """
    num_states = len(ranks)
    obs_dim = emissions.shape[-1]
    key = jr.PRNGKey(0)

    # 1. Initialize factors for each discrete state
    # Each state 'k' has H_k = A_k @ B_k
    factors_A = [jr.normal(jr.PRNGKey(i), (obs_dim, r)) for i, r in enumerate(ranks)]
    factors_B = [jr.normal(jr.PRNGKey(i+10), (r, state_dim)) for i, r in enumerate(ranks)]
    
    # Initialize HMM transition matrix and initial probabilities
    log_probs = jr.normal(key, (num_states, num_states))
    transitions = jax.nn.softmax(log_probs, axis=-1)

    # 2. Loss function using Marginal Log-Likelihood
    # In an SLDS, we approximate the likelihood of the current observation 
    # being generated by a specific rank-state.
    def loss_fn(params, emissions):
        A_factors, B_factors, trans_mat = params
        
        # Calculate individual state log-likelihoods
        state_lls = []
        for k in range(num_states):
            H_k = A_factors[k] @ B_factors[k]
            # Use Dynamax's internal LG-SSM for each mode's likelihood
            model = LinearGaussianSSM(state_dim, obs_dim)
            # (Simplified: assumes other parameters are fixed or also learned)
            ll = model.marginal_log_prob(model.initialize(key)[0], emissions)
            state_lls.append(ll)
            
        # Combine using the HMM transition logic (Forward algorithm)
        # Note: In practice, use Dynamax SwitchingLDS.marginal_log_prob
        return -jnp.sum(jnp.stack(state_lls)) 

    # 3. Optimize via Optax
    optimizer = optax.adam(1e-3)
    trainable_params = (factors_A, factors_B, transitions)
    opt_state = optimizer.init(trainable_params)

    @jit
    def step(params, opt_state):
        loss, grads = value_and_grad(loss_fn)(params, emissions)
        updates, opt_state = optimizer.update(grads, opt_state)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss

    # Training loop...
    return trainable_params
```

This code has some bugs, but it will serve as my starting point

In [2]:
import jax.nn as jnn
import jax.numpy as jnp
import jax.random as jr
from jax import jit, value_and_grad
import optax
from dynamax.linear_gaussian_ssm import LinearGaussianSSM

def train_switching_low_rank_lds(emissions, state_dim, ranks, num_iters=300):
    """
    emissions: (T, obs_dim)
    ranks: List of ranks for each discrete state (e.g., [2, 5, 8])
    """
    num_states = len(ranks)
    obs_dim = emissions.shape[-1]
    key = jr.PRNGKey(0)

    # 1. Initialize factors for each discrete state
    # Each state 'k' has H_k = A_k @ B_k
    factors_A = [jr.normal(jr.PRNGKey(i), (obs_dim, r)) for i, r in enumerate(ranks)]
    factors_B = [jr.normal(jr.PRNGKey(i+10), (r, state_dim)) for i, r in enumerate(ranks)]
    
    # Initialize HMM transition matrix and initial probabilities
    log_probs = jr.normal(key, (num_states, num_states))
    transitions = jnn.softmax(log_probs, axis=-1)

    # 2. Loss function using Marginal Log-Likelihood
    # In an SLDS, we approximate the likelihood of the current observation 
    # being generated by a specific rank-state.
    def loss_fn(params, emissions):
        A_factors, B_factors, trans_mat = params
        
        # Calculate individual state log-likelihoods
        state_lls = []
        for k in range(num_states):
            H_k = A_factors[k] @ B_factors[k]
            # Use Dynamax's internal LG-SSM for each mode's likelihood
            model = LinearGaussianSSM(state_dim, obs_dim)
            # (Simplified: assumes other parameters are fixed or also learned)
            ll = model.marginal_log_prob(model.initialize(key)[0], emissions)
            state_lls.append(ll)
            
        # Combine using the HMM transition logic (Forward algorithm)
        # Note: In practice, use Dynamax SwitchingLDS.marginal_log_prob
        return -jnp.sum(jnp.stack(state_lls)) 

    # 3. Optimize via Optax
    optimizer = optax.adam(1e-3)
    trainable_params = (factors_A, factors_B, transitions)
    opt_state = optimizer.init(trainable_params)

    @jit
    def step(params, opt_state):
        loss, grads = value_and_grad(loss_fn)(params, emissions)
        updates, opt_state = optimizer.update(grads, opt_state)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss

    # Training loop...
    
    
    
    return trainable_params

/Users/tomstone/miniforge3/envs/dynamax_env/lib/python3.12/site-packages/fastprogress/fastprogress.py:171: UserWarning: Couldn't import ipython display functions, progress bar will use console behavior
  warn("Couldn't import ipython display functions, progress bar will use console behavior")


In [3]:
state_dim = 20
obs_dim = 128
max_dim = 20

model = LinearGaussianSSM(state_dim, obs_dim)
params, _ = model.initialize()

In [4]:
x = np.random.randn(obs_dim, state_dim) * 0.001
svd = jnp.linalg.svd(x)

trainable_params = {'U' : svd.U[:, :max_dim], 'Vh' : svd.Vh[:max_dim, :], 'sigma' : svd.S[:max_dim], 'other_params' : params}

In [5]:
def loss_fn(trainable_params, batch_emissions, l = 1000):
    C_low_rank = trainable_params['U'] @ jnp.diag(jnp.abs(trainable_params['sigma'])) @ trainable_params['Vh']
    
    p = trainable_params['other_params']
    
    q = p.emissions._replace(weights = C_low_rank)
    
    p._replace(emissions = q)
    # p = p._replace(emissions_weights = C_low_rank)
    
    return -model.marginal_log_prob(p, batch_emissions) + l * jnp.abs(trainable_params['sigma'].sum())
    

In [6]:
loss_fn(trainable_params, jnp.zeros((1280, 128)))

Array(23139.154, dtype=float32)

In [7]:
optimizer = optax.adam(learning_rate=1e-3)
opt_state = optimizer.init(trainable_params)

In [8]:
@jit
def train_step(trainable_params, opt_state, batch_emissions, max_dim = 20):
    # value_and_grad gives both the scalar loss and the gradient pytree
    loss, grads = value_and_grad(loss_fn)(trainable_params, batch_emissions)
    
    # Transform gradients into updates (e.g., scaling by learning rate)
    updates, opt_state = optimizer.update(grads, opt_state)
    
    # Apply updates to the parameters
    ps = optax.apply_updates(trainable_params, updates)
    
    svd = jnp.linalg.svd(ps['other_params'].emissions.weights)
    
    ps['U'] = svd.U[:,:max_dim]
    ps['Vh'] = svd.Vh[:, :max_dim]
    ps['sigma'] = svd.S[:max_dim]
    
    return ps, opt_state, loss


In [9]:
trainable_params['sigma']

Array([0.01482937, 0.01445086, 0.01380473, 0.01324145, 0.01306993,
       0.01209727, 0.01204937, 0.0117006 , 0.01154154, 0.01117295,
       0.01068471, 0.01061252, 0.01007153, 0.00982334, 0.00930331,
       0.00878912, 0.00871953, 0.00821324, 0.00775839, 0.00715289],      dtype=float32)

In [13]:
a, b, c = train_step(trainable_params, opt_state, np.random.randn(1000, 128) * 0.1)

In [11]:
trainable_params['sigma']

Array([0.01482937, 0.01445086, 0.01380473, 0.01324145, 0.01306993,
       0.01209727, 0.01204937, 0.0117006 , 0.01154154, 0.01117295,
       0.01068471, 0.01061252, 0.01007153, 0.00982334, 0.00930331,
       0.00878912, 0.00871953, 0.00821324, 0.00775839, 0.00715289],      dtype=float32)

In [14]:
a, b, c = train_step(a, opt_state, np.random.randn(1000, 128))
a

{'U': Array([[-0.05806943, -0.13233277, -0.01088462, ...,  0.03934505,
          0.00115004,  0.05807452],
        [-0.12880854,  0.06063851,  0.14205581, ...,  0.05818253,
         -0.03576558, -0.05809202],
        [-0.15963611,  0.00417515, -0.16497818, ...,  0.00017378,
         -0.06127198, -0.01247894],
        ...,
        [ 0.11821492,  0.03736878,  0.1099209 , ...,  0.17223103,
          0.03176753,  0.08679888],
        [-0.07666431,  0.06317952, -0.01077211, ..., -0.0748493 ,
         -0.1399147 ,  0.08972174],
        [ 0.01284755,  0.00648827, -0.17234574, ..., -0.09789759,
          0.04654662, -0.00237845]], dtype=float32),
 'Vh': Array([[ 3.88801023e-02,  2.50548244e-01,  2.86624134e-01,
          3.54901925e-02,  8.97693858e-02,  4.21799809e-01,
         -3.39387596e-01,  1.26729682e-01,  4.22213823e-02,
          2.77879745e-01,  6.64538965e-02, -4.22569290e-02,
          8.64048302e-02,  2.76540577e-01,  3.61006737e-01,
         -2.91988671e-01, -3.82611781e-01, -6.4

In [15]:
import pickle as pkl

In [20]:
with open('../../gdrive/mc_pacman.pkl', 'rb') as f:
    p = pkl.load(f)

Had google AI convert my Julia code to python

In [27]:
def assemble_data(p, delta_t=20, dtype=np.float64):
    data = {
        "spike": {},
        "force": {}
    }

    for i in range(1, 9):
        # Create boolean mask for the condition
        logics = np.array(p["condition"]) == i
        
        if not np.any(logics):
            continue
            
        # Find indices where condition is true
        logic_indices = np.where(logics)[0]
        
        # Get dimensions from the first matching entry
        first_ind = logic_indices[0]
        spike_shape = p["spikes"][first_ind].shape  # (features, time)
        num_trials = np.sum(logics)
        
        N = spike_shape[1] // delta_t
        
        # Initialize arrays: (features, N, trials)
        data["spike"][i] = np.empty((spike_shape[0], N, num_trials), dtype=dtype)
        data["force"][i] = np.empty((1, N, num_trials), dtype=dtype)

        for j, ind in enumerate(logic_indices):
            mat1 = p["spikes"][ind]
            mat2 = p["force"][ind]

            for k in range(N):
                start = delta_t * k
                end = delta_t * (k + 1)
                
                # sum and mean across the time window (axis 1)
                data["spike"][i][:, k, j] = np.sum(mat1[:, start:end], axis=1)
                data["force"][i][:, k, j] = np.mean(mat2[:, start:end], axis=1)

    return data


In [25]:
data = assemble_data(p)

In [31]:
data['spike'][1].shape

(128, 300, 38)

In [32]:
for _ in range(300):
    trainable_params, b, c = train_step(trainable_params, opt_state, data['spike'][1][:, :, np.random.randint(0, data['spike'][1].shape[2])].T)
    